<a href="https://colab.research.google.com/github/shace-tariq/flyrank/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shace-tariq/flyrank/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
# Your experiment here
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Helper function
# ------------------------------------------------------------

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()


# ------------------------------------------------------------
# 2. Experiment A — Change max_depth to 3 and 4
# ------------------------------------------------------------

features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining_label"].values

print("=" * 60)
print("EXPERIMENT A — DECISION TREE DEPTH")
print("=" * 60)

depth_results = []

for depth in [2, 3, 4]:

    model = DecisionTreeClassifier(
        max_depth=depth,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X, y)

    scores = model.predict_proba(X)[:, 1]

    p20 = precision_at_k(scores, y, 20)
    p50 = precision_at_k(scores, y, 50)

    depth_results.append({
        "max_depth": depth,
        "Precision@20": p20,
        "Precision@50": p50
    })

    print(f"\n--- max_depth = {depth} ---")
    print(f"Precision@20: {p20:.3f}")
    print(f"Precision@50: {p50:.3f}")

    print("\nReadable tree:")
    print(export_text(model, feature_names=features))

print("\nDepth comparison:")
print(pd.DataFrame(depth_results).round(3))


# ------------------------------------------------------------
# 3. Experiment B — Change the features
#    Drop impressions_90d and add engagement_rate
# ------------------------------------------------------------

new_features = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

X_new = (
    df[new_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("\n" + "=" * 60)
print("EXPERIMENT B — DIFFERENT FEATURES")
print("=" * 60)

tree_new = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_new.fit(X_new, y)

new_scores = tree_new.predict_proba(X_new)[:, 1]

print("\nTree after dropping impressions_90d")
print("and adding engagement_rate:\n")

print(export_text(
    tree_new,
    feature_names=new_features
))

print(
    f"\nModified tree Precision@50: "
    f"{precision_at_k(new_scores, y, 50):.3f}"
)

print(
    "\nFirst split chosen by the tree:"
)

print(
    export_text(
        tree_new,
        feature_names=new_features,
        max_depth=1
    )
)


# ------------------------------------------------------------
# 4. Experiment C — Client-holdout train/test validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXPERIMENT C — CLIENT-HOLDOUT VALIDATION")
print("=" * 60)

# Try to identify the client/group column automatically.
possible_group_columns = [
    "client_id",
    "client",
    "site_id",
    "site",
    "domain"
]

group_column = None

for col in possible_group_columns:
    if col in df.columns:
        group_column = col
        break

if group_column is None:
    print("\nCould not automatically find the client column.")
    print("Available columns are:")
    print(df.columns.tolist())
else:

    print(f"\nUsing '{group_column}' as the client/group column.")

    groups = df[group_column]

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(X, y, groups=groups)
    )

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    print(f"Training rows: {len(train_idx)}")
    print(f"Testing rows:  {len(test_idx)}")

    print(
        f"Training clients: "
        f"{groups.iloc[train_idx].nunique()}"
    )

    print(
        f"Testing clients:  "
        f"{groups.iloc[test_idx].nunique()}"
    )

    # Train tree ONLY on training clients
    holdout_tree = DecisionTreeClassifier(
        max_depth=2,
        class_weight="balanced",
        random_state=42
    )

    holdout_tree.fit(X_train, y_train)

    # Predict on completely unseen clients
    test_scores = holdout_tree.predict_proba(X_test)[:, 1]

    holdout_precision = precision_at_k(
        test_scores,
        y_test,
        50
    )

    print(
        f"\nClient-holdout Tree Precision@50: "
        f"{holdout_precision:.3f}"
    )

    print("\nTree learned from training clients:")
    print(
        export_text(
            holdout_tree,
            feature_names=features
        )
    )


# ------------------------------------------------------------
# 5. Compare hand rule vs tree on the same test set
# ------------------------------------------------------------

if group_column is not None:

    test_hand_rule = df.iloc[test_idx]["hand_rule_score"].values

    hand_precision = precision_at_k(
        test_hand_rule,
        y_test,
        50
    )

    print("\n" + "=" * 60)
    print("FINAL COMPARISON — UNSEEN CLIENTS")
    print("=" * 60)

    print(
        f"Hand rule Precision@50: "
        f"{hand_precision:.3f}"
    )

    print(
        f"Tree Precision@50:      "
        f"{holdout_precision:.3f}"
    )

    print(
        f"\nTree - Hand rule: "
        f"{holdout_precision - hand_precision:+.3f}"
    )


EXPERIMENT A — DECISION TREE DEPTH

--- max_depth = 2 ---
Precision@20: 0.550
Precision@50: 0.600

Readable tree:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0


--- max_depth = 3 ---
Precision@20: 0.700
Precision@50: 0.720

Readable tree:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- conte

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.